## Data Preprocessing

In [ ]:
from datasets import load_dataset
from PIL import Image
from transformers import BertTokenizer
import matplotlib.pyplot as plt

# Load the dataset (select only the first sample for this example)
flickr_dataset = load_dataset("nlphuji/flickr30k", split="test").select(range(1))

# Extract the first image and its captions
try:
    first_image = flickr_dataset[0]["image"]
    first_image_captions = flickr_dataset[0]["caption"]
except Exception as e:
    print(f"Error accessing data: {e}")
    exit()

# Print the captions
print("Original Captions:")
for cap in first_image_captions:
    print("- ", cap)

# Display the image
plt.imshow(first_image)
plt.axis("off")
plt.title("First Image from Dataset")
plt.show()

# --- Now, let's simulate what the ImageCaptionDataset and DataLoader would do ---

# 1. Initialize Tokenizer (same as in your main code)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# 2. Build a simplified Vocabulary (for demonstration purposes)
# In your actual code, you build a more sophisticated vocabulary, but here we'll just
# manually create a small one to illustrate the process.
class SimpleVocab:
    def __init__(self, tokenizer):
        self.word2idx = {}
        self.idx2word = {}
        self.idx = 0
        self.tokenizer = tokenizer

        # Add special tokens
        for special_token in [tokenizer.pad_token, tokenizer.unk_token, tokenizer.cls_token, tokenizer.sep_token]:
            if special_token is not None:
                self.add_word(special_token)

    def add_word(self, word):
        if word not in self.word2idx:
            self.word2idx[word] = self.idx
            self.idx2word[self.idx] = word
            self.idx += 1

    def __call__(self, word):
        if word not in self.word2idx:
            return self.word2idx.get(self.tokenizer.unk_token, 0)
        return self.word2idx[word]
    
    def __len__(self):
        return len(self.word2idx)

# Create an instance of the simple vocabulary
vocab = SimpleVocab(tokenizer)

# Manually add some words from the first image's captions to the vocabulary
for cap in first_image_captions:
    tokens = tokenizer.tokenize(cap.lower())
    for token in tokens:
        vocab.add_word(token)


# 3. Simulate ImageCaptionDataset's __getitem__
def simulate_getitem(image, caption_list, tokenizer, vocab):
    # Tokenize and vectorize caption
    combined_caption = " ".join(caption_list)
    tokens = tokenizer.tokenize(str(combined_caption).lower())
    caption_vec = [vocab(tokenizer.cls_token)]
    caption_vec.extend([vocab(token) for token in tokens])
    caption_vec.append(vocab(tokenizer.sep_token))

    print("\nSimulated Tokenized Caption (using SimpleVocab):")
    print(caption_vec)

    print("\nCorresponding Words (using SimpleVocab):")
    print([vocab.idx2word.get(idx, tokenizer.unk_token) for idx in caption_vec])

    return image, caption_vec

# Call the simulation function
simulated_image, simulated_caption_vec = simulate_getitem(
    first_image, first_image_captions, tokenizer, vocab
)

# 4. Display the image again (for verification)
plt.imshow(simulated_image)
plt.axis("off")
plt.title("Image (after simulated __getitem__)")
plt.show()

print("\nIf the printed captions, tokenized captions, corresponding words, and the two images match,")
print("then it's a good indication that your ImageCaptionDataset and DataLoader are likely parsing")
print("the images and captions correctly.")

## Model creation

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
from datasets import load_dataset
import math
from torch.nn.utils.rnn import pad_sequence
from transformers import BertTokenizer, ViTModel, BertModel
import matplotlib.pyplot as plt

# --- Data Loading and Preprocessing ---


def custom_collate_fn(batch):
    images, captions = zip(*batch)

    # Stack images into a batch tensor
    images = torch.stack(images)

    # Pad captions to the same length
    captions = [torch.tensor(caption) for caption in captions]
    captions_padded = pad_sequence(captions, batch_first=True, padding_value=0)  # Use padding token index (e.g., 0)

    # Create caption mask
    caption_mask = (captions_padded != 0)  # True for valid tokens, False for padding

    return images, captions_padded, caption_mask


class ImageCaptionDataset(Dataset):
    def __init__(self, images, captions, tokenizer, vocab, transform=None):
        self.images = images
        self.captions = captions
        self.tokenizer = tokenizer
        self.vocab = vocab
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Preprocess image
        image = self.images[idx].convert('RGB')
        if self.transform:
            image = self.transform(image)

        # Tokenize and vectorize caption
        caption_list = self.captions[idx]
        combined_caption = " ".join(caption_list)
        tokens = self.tokenizer.tokenize(str(combined_caption).lower())
        caption_vec = [self.vocab(self.tokenizer.cls_token)]
        caption_vec.extend([self.vocab(token) for token in tokens])
        caption_vec.append(self.vocab(self.tokenizer.sep_token))
        target = torch.LongTensor(caption_vec)

        return image, target


class Vocabulary:
    def __init__(self, tokenizer):
        self.word2idx = {}
        self.idx2word = {}
        self.idx = 0
        self.tokenizer = tokenizer  # Store the tokenizer

    def add_word(self, word):
        if word not in self.word2idx:
            self.word2idx[word] = self.idx
            self.idx2word[self.idx] = word
            self.idx += 1

    def __call__(self, word):
        if word not in self.word2idx:
            return self.word2idx.get(self.tokenizer.unk_token, 0)  # Use tokenizer's unk_token
        return self.word2idx[word]

    def __len__(self):
        return len(self.word2idx)


def build_vocab(captions_list, threshold=1, tokenizer=None):
    if tokenizer is None:
        raise ValueError("A tokenizer must be provided to build the vocabulary.")

    # Count the frequency of each token
    counter = {}
    for captions in captions_list:
        for caption in captions:
            tokens = tokenizer.tokenize(str(caption).lower())
            for word in tokens:
                counter[word] = counter.get(word, 0) + 1

    # Initialize the vocabulary
    vocab = Vocabulary(tokenizer)

    # Add special tokens from the tokenizer
    for special_token in [tokenizer.pad_token, tokenizer.unk_token, tokenizer.cls_token, tokenizer.sep_token]:
        if special_token is not None:
            vocab.add_word(special_token)

    # Add words that meet the frequency threshold
    words = [word for word, cnt in counter.items() if cnt >= threshold]
    for word in words:
        if word not in vocab.word2idx:  # Avoid duplicates
            vocab.add_word(word)

    return vocab


class EncoderViT(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        self.config = self.vit.config
        self.linear = nn.Linear(self.config.hidden_size, embed_size)
        self.bn = nn.BatchNorm1d(embed_size, momentum=0.01)

        for param in self.vit.parameters():
            param.requires_grad = False

    def forward(self, images):
        with torch.no_grad():
            features = self.vit(images).last_hidden_state
        features = features[:, 0, :]
        features = self.bn(self.linear(features))
        return features


class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, dropout_p=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout_p)

        pe = torch.zeros(max_len, embed_size)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_size, 2).float() * (-math.log(10000.0) / embed_size))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)